In [7]:
import pandas as pd
import numpy as np

pd.options.display.float_format = '${:,.2f}'.format

### Steps to calculate FRTB regulatory charge:
We focus on **Equity Delta Risk**, incorporating:
 1. **Issuer-level Netting** (netting long/short positions on the same underlying name).
 2. **Risk-Weighting** (applying regulatory risk weights based on equity classification/buckets).
 3. **Intra-Bucket Aggregation** (applying correlation metrics between different issuers in the same bucket to find $K_b$).
 4. **Inter-Bucket Aggregation** (aggregating across buckets using cross-bucket correlation $\gamma_{bc}$ for the final capital charge).

### Model inputs:
1. `Risk weights`: These are the weights given to different sectors like technology, financials, consumers etc. which depict the riskiness associated with these sectors. These weights are provided by the regulator.
2. `Intra bucket correlation parameters`: These are parameters which reflect the amount of correlation that exists within a certain sector (bucket).
3. `Inter bubket correlation parameter`: This parameter reflect the amount of correlation that exist across two different sector (buckets) and are provided by the regulatory guidelines.
4. `Portfolio data`: The data which consists following information:
    - `issuer`: The company whose equity is present in the portfolio
    - `ticker`: The symbol used by the equity
    - `bucket`: The sector (bucket) which the company belongs to
    - `sensitivity`: The delta sensitivity


In [2]:
risk_weights = {
    1: 0.30,  # Technology, Telecom (Large Cap)
    2: 0.35,  # Financials, Industrials (Large Cap)
    3: 0.30,  # Consumer, Utilities (Large Cap)
}

In [3]:
# Intra-bucket correlation parameters (rho)
intra_correlations = {
    1: 0.25,  # 25% correlation within tech
    2: 0.25,  # 25% correlation within financials
    3: 0.25,  # 25% correlation within consumer
}

In [4]:
# Inter-bucket correlation parameter (gamma) - flat 15% as per standard guidelines
gamma_flat = 0.15

In [5]:
# Input Portfolio: Raw Delta Sensitivities (in USD)
portfolio_data = [
    {"issuer": "Apple Inc.", "ticker": "AAPL", "bucket": 1, "sensitivity": 1200000},
    {"issuer": "Apple Inc.", "ticker": "AAPL_Option", "bucket": 1, "sensitivity": -200000}, # Netting demo
    {"issuer": "Microsoft Corp.", "ticker": "MSFT", "bucket": 1, "sensitivity": 800000},
    {"issuer": "JPMorgan Chase", "ticker": "JPM", "bucket": 2, "sensitivity": 1500000},
    {"issuer": "Goldman Sachs", "ticker": "GS", "bucket": 2, "sensitivity": -600000},   # Hedging demo
    {"issuer": "Amazon.com Inc.", "ticker": "AMZN", "bucket": 3, "sensitivity": 500000},
]

In [8]:
df_portfolio = pd.DataFrame(portfolio_data)
df_portfolio

,issuer,ticker,bucket,sensitivity
0,Apple Inc.,AAPL,1,1200000
1,Apple Inc.,AAPL_Option,1,-200000
2,Microsoft Corp.,MSFT,1,800000
3,JPMorgan Chase,JPM,2,1500000
4,Goldman Sachs,GS,2,-600000
5,Amazon.com Inc.,AMZN,3,500000


### Step 1. Issuer level Netting

In [17]:
df_netted = df_portfolio.groupby(['issuer', 'bucket'])['sensitivity'].sum().reset_index()
df_netted

,issuer,bucket,sensitivity
0,Amazon.com Inc.,3,500000
1,Apple Inc.,1,1000000
2,Goldman Sachs,2,-600000
3,JPMorgan Chase,2,1500000
4,Microsoft Corp.,1,800000


### Step 2. Applying risk weights

In [18]:
df_netted['risk_weight'] = df_netted['bucket'].map(risk_weights)
df_netted

,issuer,bucket,sensitivity,risk_weight
0,Amazon.com Inc.,3,500000,$0.30
1,Apple Inc.,1,1000000,$0.30
2,Goldman Sachs,2,-600000,$0.35
3,JPMorgan Chase,2,1500000,$0.35
4,Microsoft Corp.,1,800000,$0.30


In [19]:
df_netted['weighted_sensitivity'] = df_netted['sensitivity'] * df_netted['risk_weight']
df_netted

,issuer,bucket,sensitivity,risk_weight,weighted_sensitivity
0,Amazon.com Inc.,3,500000,$0.30,"$150,000.00"
1,Apple Inc.,1,1000000,$0.30,"$300,000.00"
2,Goldman Sachs,2,-600000,$0.35,"$-210,000.00"
3,JPMorgan Chase,2,1500000,$0.35,"$525,000.00"
4,Microsoft Corp.,1,800000,$0.30,"$240,000.00"


### Step 3. Intra-bucket aggregation

In [ ]:
bucket_charges = {}
bucket_sum_weighted_sensitivities = {}
S_b = {}

for bucket, group in df_netted.groupby(['bucket']):
    ws = group['weighted_sensitivity'].values
    rho = intra_correlations.get(bucket, 0.25)
    corr_matrix = np.full((len(ws), len(ws)), rho)
    corr_matrix[np.diag_indices_from(corr_matrix)] = 1
    bucket_charges[bucket] = np.sqrt(max(0, ws.T @ corr_matrix @ ws))
    bucket_sum_weighted_sensitivities[bucket] = ws.sum()

    print(f"\n  Bucket {bucket} Details:")
    print(f"    - Issuers in bucket: {list(group['issuer'])}")
    print(f"    - Weighted Sensitivities: {['${:,.2f}'.format(x) for x in ws]}")
    print(f"    - Correlation parameter (rho): {rho * 100}%")
    print(f"    - Bucket Capital Charge (Kb): ${bucket_charges[bucket]:,.2f}")

for bucket in bucket_charges:
    Kb = bucket_charges[bucket]
    ws = bucket_sum_weighted_sensitivities[bucket]
    S_b[bucket] = max(-Kb, min(Kb, ws)) # Capping rule: Sb is bounded by [-Kb, Kb]
    print(f"\n  Bucket {bucket} Final Charge (S_b): ${S_b[bucket]:,.2f}")



  Bucket (1,) Details:
    - Issuers in bucket: ['Apple Inc.', 'Microsoft Corp.']
    - Weighted Sensitivities: ['$300,000.00', '$240,000.00']
    - Correlation parameter (rho): 25.0%
    - Bucket Capital Charge (Kb): $428,485.71

  Bucket (2,) Details:
    - Issuers in bucket: ['Goldman Sachs', 'JPMorgan Chase']
    - Weighted Sensitivities: ['$-210,000.00', '$525,000.00']
    - Correlation parameter (rho): 25.0%
    - Bucket Capital Charge (Kb): $514,392.85

  Bucket (3,) Details:
    - Issuers in bucket: ['Amazon.com Inc.']
    - Weighted Sensitivities: ['$150,000.00']
    - Correlation parameter (rho): 25.0%
    - Bucket Capital Charge (Kb): $150,000.00

  Bucket (1,) Final Charge (S_b): $428,485.71

  Bucket (2,) Final Charge (S_b): $315,000.00

  Bucket (3,) Final Charge (S_b): $150,000.00


### Step 4. Inter bucket aggregation and final capital charge calculation

In [30]:
sum_Kb_sq = sum(Kb**2 for Kb in bucket_charges.values())
active_buckets = list(bucket_charges.keys())
cross_terms = 0.0
for i in range(len(bucket_charges)):
    for j in range(len(bucket_charges)):
        if i != j:
            b = active_buckets[i]
            c = active_buckets[j]
            cross_terms += gamma_flat * S_b[b] * S_b[c]
total_SBM_charge = np.sqrt(max(0, sum_Kb_sq + cross_terms))
print(f"\nTotal SBM Capital Charge: ${total_SBM_charge:,.2f}")




Total SBM Capital Charge: $738,003.22


In [31]:
### Gross unhedged capital charge
gross_unhedged_charge = sum(abs(s) * risk_weights[b] for s, b in zip(df_portfolio['sensitivity'], df_portfolio['bucket']))
print(f"\nGross Unhedged Capital Charge: ${gross_unhedged_charge:,.2f}")


Gross Unhedged Capital Charge: $1,545,000.00


In [32]:
### SBM Efficiency
sbm_efficiency = (gross_unhedged_charge - total_SBM_charge) / gross_unhedged_charge * 100
print(f"SBM Efficiency: {sbm_efficiency:.2f}%")

SBM Efficiency: 52.23%
